# Pay Disparity

This notebook looks at one question inside UK energy and utilities companies:

**how has CEO pay moved compared with median employee pay over time?**

The project uses public UK pay-ratio disclosures, filters the dataset to energy-related firms, keeps only companies with enough history to compare across years, and then re-indexes pay series to a common starting point of `100`.

That makes the notebook less about absolute salary size and more about movement, divergence, and trend.

## Why this framing?

A CEO can obviously earn far more than a median employee in absolute terms. But if the goal is to study inequality over time, the more useful question is whether those two lines are changing at the same speed.

Re-indexing both series to `100` in the first available year gives a cleaner comparison:

- `150` means a series is `50%` above its starting point
- `80` means it is `20%` below its starting point
- the gap between the two lines shows whether executive and worker pay are drifting apart

In [ ]:
import pandas as pd

wages = pd.read_csv('data/cleaned/wages.csv')
companies = pd.read_csv('data/cleaned/companies.csv')

print('Companies in final sample:', wages['Company'].nunique())
print('Rows in cleaned dataset:', len(wages))
print('Coverage:', int(wages['Year'].min()), 'to', int(wages['Year'].max()))

companies

## Building a simple comparison table

For each company, I compare the first and last available years in the cleaned sample and measure how much indexed CEO pay and indexed median pay changed over that window.

In [ ]:
rows = []

for company, group in wages.groupby('Company'):
    group = group.sort_values('Year').copy()
    ceo_start = group['ceo_pay_k'].iloc[0] * 1000
    ceo_end = group['ceo_pay_k'].iloc[-1] * 1000
    median_start = group['m_pay'].iloc[0]
    median_end = group['m_pay'].iloc[-1]

    ceo_index_end = (ceo_end / ceo_start) * 100
    median_index_end = (median_end / median_start) * 100

    rows.append(
        {
            'Company': company,
            'Start year': int(group['Year'].iloc[0]),
            'End year': int(group['Year'].iloc[-1]),
            'CEO change %': round((ceo_end / ceo_start - 1) * 100, 1),
            'Median change %': round((median_end / median_start - 1) * 100, 1),
            'Indexed gap (points)': round(ceo_index_end - median_index_end, 1),
        }
    )

summary = pd.DataFrame(rows).sort_values('Indexed gap (points)', ascending=False)
summary

## What stands out

A few patterns jump out from the cleaned sample:

- **Centrica** is the clearest case of divergence. CEO pay is up by roughly `264%` from `2019` to `2024`, while median pay is up by roughly `35%`.
- **Drax** also shows a strong split, with CEO pay growing much faster than worker pay.
- The pattern is not universal. **BP** and **Royal Dutch Shell** both show cases where median pay rose while indexed CEO pay ended below its own starting level.

That is what made this project feel more interesting than a simple anti-executive chart dump. The data shows divergence in some firms, but not in all of them.

## Example charts

### BP
![BP chart](data/cleaned/charts/bp_indexed_pay.png)

### Centrica
![Centrica chart](data/cleaned/charts/centrica_indexed_pay.png)

### Drax
![Drax chart](data/cleaned/charts/drax_indexed_pay.png)

## Project pipeline

The repo is deliberately simple:

1. `scripts/dataCleaner.py` cleans the raw pay-ratio dataset and filters it to energy and utilities companies
2. `scripts/plotIndexedPayCharts.py` generates one indexed chart per company
3. `data/cleaned/` stores the final analysis-ready CSVs and chart outputs

To rebuild everything:

```python
python scripts/dataCleaner.py
python scripts/plotIndexedPayCharts.py
```